# Presentación de Datos – ICFES Saber 11

Este notebook carga el dataset original desde Kaggle, realiza la preparación inicial de columnas
y genera el dataset limpio (`v2_datos_limpios.csv`) que se sube a Kaggle para ser usado
en los notebooks de modelado y estudio exploratorio.

**Decisiones de limpieza documentadas en:** `feature_selection.md`

## 0. Dependencias

In [8]:
#%pip install -q "kagglehub[pandas-datasets]" pandas

## 1. Descarga y carga desde Kaggle

Requiere credenciales en `~/.kaggle/kaggle.json`.  
Obtenerlas en **https://www.kaggle.com/settings → API → Create New Token**.

In [9]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "jacoboayala/datos-icfes-11",
    "v1_todos_los_datos.csv",
    pandas_kwargs={"sep": ";", "low_memory": False}
)

print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head(3)

Filas: 4,821,037  |  Columnas: 55


,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,cole_cod_mcpio_ubicacion,cole_codigo_icfes,cole_depto_ubicacion,...,fami_tieneinternet,fami_tienelavadora,fami_tieneserviciotv,periodo,punt_c_naturales,punt_global,punt_ingles,punt_lectura_critica,punt_matematicas,punt_sociales_ciudadanas
0,RURAL,N,A,TÉCNICO/ACADÉMICO,276109000243,276109000243,76,76109,95125,VALLE,...,No,No,No,20142,53,274,41,61,47,63
1,URBANO,N,A,TÉCNICO/ACADÉMICO,227361000816,227361000816,27,27450,35915,CHOCO,...,No,Si,No,20142,43,222,48,40,45,48
2,URBANO,N,A,TÉCNICO/ACADÉMICO,150001003081,150001003081,50,50001,61093,META,...,Si,Si,Si,20142,71,358,62,72,74,73


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4821037 entries, 0 to 4821036
Data columns (total 55 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   cole_area_ubicacion            object 
 1   cole_bilingue                  object 
 2   cole_calendario                object 
 3   cole_caracter                  object 
 4   cole_cod_dane_establecimiento  int64  
 5   cole_cod_dane_sede             int64  
 6   cole_cod_depto_ubicacion       int64  
 7   cole_cod_mcpio_ubicacion       int64  
 8   cole_codigo_icfes              int64  
 9   cole_depto_ubicacion           object 
 10  cole_genero                    object 
 11  cole_jornada                   object 
 12  cole_mcpio_ubicacion           object 
 13  cole_naturaleza                object 
 14  cole_nombre_establecimiento    object 
 15  cole_nombre_sede               object 
 16  cole_sede_principal            object 
 17  desemp_ingles                  object 
 18  es

## 2. Preparación de datos

In [11]:
# Convertir puntajes a numérico
PUNTAJES = ["punt_global", "punt_matematicas", "punt_lectura_critica",
            "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles"]

for col in PUNTAJES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Descomponer periodo en año y semestre
df["periodo"] = df["periodo"].astype(str)
df["anio"] = df["periodo"].str[:4].astype(int)
df["semestre"] = df["periodo"].str[4:]

# Normalizar etiquetas de texto
df["cole_area_ubicacion"] = df["cole_area_ubicacion"].str.strip().str.title()
df["cole_naturaleza"]     = df["cole_naturaleza"].str.strip().str.title()
df["estu_genero"]         = df["estu_genero"].str.strip().str.upper()

print("Periodos:", sorted(df["periodo"].unique()))
print("Área:    ", df["cole_area_ubicacion"].value_counts().to_dict())
print("Naturaleza:", df["cole_naturaleza"].value_counts().to_dict())

Periodos: ['20142', '20151', '20152', '20161', '20162', '20171', '20172', '20181', '20182', '20191', '20192', '20201', '20202', '20211', '20212', '20221', '20222', '20231', '20232', '20241', '20242']
Área:     {'Urbano': 4185434, 'Rural': 635603}
Naturaleza: {'Oficial': 3671613, 'No Oficial': 1149424}


## 3. Selección de variables y generación del dataset limpio

Las justificaciones completas de cada eliminación están en `feature_selection.md`.

In [12]:
# 1. Data leakage: sub-puntajes son componentes directos de punt_global
LEAKAGE = [
    "punt_matematicas", "punt_lectura_critica", "punt_c_naturales",
    "punt_sociales_ciudadanas", "punt_ingles", "desemp_ingles"
]

# 2. Identificadores administrativos: no tienen valor predictivo
IDENTIFICADORES = [
    "estu_estudiante", "estu_consecutivo",
    "cole_cod_dane_establecimiento", "cole_cod_dane_sede",
    "cole_cod_depto_ubicacion", "cole_cod_mcpio_ubicacion",
    "cole_codigo_icfes",
    "estu_cod_depto_presentacion", "estu_cod_mcpio_presentacion",
    "estu_cod_reside_depto", "estu_cod_reside_mcpio"
]

# 3. Nombres de alta cardinalidad
ALTA_CARDINALIDAD = [
    "cole_nombre_establecimiento", "cole_nombre_sede"
]

# 4. Redundantes: la información ya existe en otras columnas
REDUNDANTES = [
    "periodo",                   # → anio + semestre
    "estu_depto_presentacion",   # → cole_depto_ubicacion
    "estu_mcpio_presentacion",   # → ídem, más alta cardinalidad
    "estu_mcpio_reside"          # → estu_depto_reside es suficiente
]

# 5. Varianza casi nula
BAJA_VARIANZA = [
    "estu_privado_libertad",  # 100% N
    "estu_pais_reside",       # 99.57% COLOMBIA
    "estu_nacionalidad"       # 99.43% COLOMBIA
]

# 6. Índices compuestos redundantes con variables fami_*
COMPUESTOS_REDUNDANTES = [
    "estu_inse_individual", "estu_nse_individual"
]

# 7. Flag administrativo derivado
FLAGS_ADMIN = ["estu_agregado"]

COLS_DROP = (
    LEAKAGE + IDENTIFICADORES + ALTA_CARDINALIDAD +
    REDUNDANTES + BAJA_VARIANZA + COMPUESTOS_REDUNDANTES +
    FLAGS_ADMIN
)

faltantes = [c for c in COLS_DROP if c not in df.columns]
if faltantes:
    print(f"⚠️  Columnas no encontradas: {faltantes}")
else:
    df_clean = df.drop(columns=COLS_DROP)
    print(f"Columnas originales : {df.shape[1]}")
    print(f"Columnas eliminadas : {len(COLS_DROP)}")
    print(f"Columnas restantes  : {df_clean.shape[1]}")
    print(f"\nFeatures finales:\n{sorted(df_clean.columns.tolist())}")

Columnas originales : 57
Columnas eliminadas : 29
Columnas restantes  : 28

Features finales:
['anio', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_depto_ubicacion', 'cole_genero', 'cole_jornada', 'cole_mcpio_ubicacion', 'cole_naturaleza', 'cole_sede_principal', 'estu_depto_reside', 'estu_fechanacimiento', 'estu_genero', 'estu_grado', 'estu_tipodocumento', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fami_tienelavadora', 'fami_tieneserviciotv', 'punt_global', 'semestre']


In [13]:
# Verificar reducción de peso en memoria
mem_original = df.memory_usage(deep=True).sum() / 1024**2
mem_clean    = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria original : {mem_original:,.1f} MB")
print(f"Memoria limpio   : {mem_clean:,.1f} MB")
print(f"Reducción        : {mem_original - mem_clean:,.1f} MB ({(1 - mem_clean/mem_original)*100:.1f}%)")

Memoria original : 12,706.9 MB
Memoria limpio   : 7,676.3 MB
Reducción        : 5,030.6 MB (39.6%)


## 4. Exportar y subir a Kaggle

In [14]:
from pathlib import Path

OUTPUT_DIR = Path("icfes_clean")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = OUTPUT_DIR / "v2_datos_limpios.csv"
df_clean.to_csv(CSV_PATH, sep=";", index=False)

size_mb = CSV_PATH.stat().st_size / 1024**2
print(f"✅ CSV guardado: {CSV_PATH}  ({size_mb:.1f} MB)")

✅ CSV guardado: icfes_clean/v2_datos_limpios.csv  (914.8 MB)


## 5. Exportar subconjuntos por área de ubicación (Urbano / Rural)

In [15]:
for area in ["Urbano", "Rural"]:
    subset = df_clean[df_clean["cole_area_ubicacion"] == area]
    filename = f"v2_datos_{area.lower()}.csv"
    path = OUTPUT_DIR / filename
    subset.to_csv(path, sep=";", index=False)
    size_mb = path.stat().st_size / 1024**2
    print(f"✅ {filename}  |  filas: {len(subset):,}  |  {size_mb:.1f} MB")

✅ v2_datos_urbano.csv  |  filas: 4,185,434  |  798.9 MB
✅ v2_datos_rural.csv  |  filas: 635,603  |  115.9 MB


In [ ]:
filas_rural = len(df_clean[df_clean["cole_area_ubicacion"] == "Rural"])
filas_urbano = len(df_clean[df_clean["cole_area_ubicacion"] == "Urbano"])
print(f"Filas en Rural: {filas_rural:,}")
print(f"Filas en Urbano: {filas_urbano:,}")
print(f"Total: {len(df_clean):,}")

print(40*"=")

print(f"Urbano: {filas_urbano / len(df_clean) * 100:.2f}%")
print(f"Rural: {filas_rural / len(df_clean) * 100:.2f}%")

print(40*"=")

Filas en Rural: 635,603
Filas en Urbano: 4,185,434
Total: 4,821,037
Urbano: 86.82%
Rural: 13.18%
